<a href="https://colab.research.google.com/github/palarunava/machine-learning-courses/blob/main/machine-learning-misc/audio_feature_extracor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from datasets import load_dataset

In [2]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

In [3]:
# model_id = "openai/whisper-large-v3-turbo"
model_id = "openai/whisper-large-v3"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

In [4]:
pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    chunk_length_s=30,
    batch_size=16,  # batch size for inference - set based on your device
    torch_dtype=torch_dtype,
    device=device,
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


In [5]:
dataset = load_dataset("distil-whisper/librispeech_long", "clean", split="validation", download_mode=False)
sample = dataset[0]["audio"]
print(sample)

README.md:   0%|          | 0.00/480 [00:00<?, ?B/s]

clean/validation-00000-of-00001-91350812(…):   0%|          | 0.00/1.98M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/1 [00:00<?, ? examples/s]

In [ ]:
result = pipe(sample)
print(result["text"])

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see re

In [ ]:
from IPython.display import display, Javascript
from google.colab import output
import base64

def record_voice(filename='recording.wav'):
  js = Javascript('''
    async function recordAudio() {
      const div = document.createElement('div');
      const button = document.createElement('button');
      button.textContent = 'Record';
      button.style.background = 'red';
      button.style.color = 'white';
      button.style.padding = '10px';
      document.body.appendChild(div);
      div.appendChild(button);

      const stream = await navigator.mediaDevices.getUserMedia({audio: true});
      const recorder = new MediaRecorder(stream);
      const chunks = [];

      recorder.ondataavailable = (e) => chunks.push(e.data);
      recorder.onstop = async () => {
        const blob = new Blob(chunks);
        const reader = new FileReader();
        reader.readAsDataURL(blob);
        reader.onloadend = () => {
          window.callback(reader.result);
        };
      };

      button.onclick = () => {
        if (recorder.state === 'inactive') {
          recorder.start();
          button.textContent = 'Stop Recording';
        } else {
          recorder.stop();
          button.textContent = 'Done!';
        }
      };

      return new Promise((resolve) => {
        window.callback = resolve;
      });
    }
  ''')
  display(js)
  data = output.eval_js('recordAudio()')
  binary = base64.b64decode(data.split(',')[1])

  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

# Run the function
audio_file = record_voice()
print(f"Saved as {audio_file}")

In [ ]:
from IPython.display import Audio
Audio(audio_file)

In [ ]:
generate_kwargs = {
    "language": "english",
    "condition_on_prev_tokens": False,
    "compression_ratio_threshold": 1.35,  # zlib compression ratio threshold (in token space)
    "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
    "logprob_threshold": -1.0,
    "no_speech_threshold": 0.6,
    "return_timestamps": True,
}

# result = pipe(sample, return_timestamps=True)
result = pipe('recording.wav', generate_kwargs=args)
print(result["text"])

In [1]:
import torch
import gc
import librosa
import numpy as np
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq
from datasets import load_dataset

# 1. Clear memory & setup hardware configuration
gc.collect()
torch.cuda.empty_cache()

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3-turbo"

# 2. Load the native model components
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True,
    attn_implementation="sdpa"
).to(device)

# 3. Pull the sample long audio array
dataset = load_dataset("distil-whisper/librispeech_long", "clean", split="validation", download_mode=False)
sr = 16000
audio_array = dataset[0]["audio"]["array"]

# 4. Define your Chunking and Stride rules
CHUNK_DURATION = 30
OVERLAP_DURATION = 5

chunk_samples = CHUNK_DURATION * sr
overlap_samples = OVERLAP_DURATION * sr
stride_samples = chunk_samples - overlap_samples

# This structure will hold your explicit chunk-wise data
chunked_results = []

print(f"Total audio length: {len(audio_array)/sr:.2f} seconds")
print("Processing explicit chunks manually...\n")

# 5. The Sliding Window Loop
for start_idx in range(0, len(audio_array), stride_samples):
    end_idx = start_idx + chunk_samples
    chunk = audio_array[start_idx:end_idx]

    # Calculate the global clock positioning for metadata records
    global_start_time = start_idx / sr
    global_end_time = min(end_idx / sr, len(audio_array) / sr)

    if len(chunk) < sr * 0.5: # Skip tiny leftover audio shards
        continue

    # Pad trailing tail chunk with zeros if it falls short of 30 seconds
    if len(chunk) < chunk_samples:
        chunk = np.pad(chunk, (0, chunk_samples - len(chunk)), 'constant')

    # FIX 1: Generate BOTH input_features and attention_mask
    inputs = processor(chunk, sampling_rate=sr, return_attention_mask=True, return_tensors="pt")
    input_features = inputs.input_features.to(device, dtype=torch_dtype)
    attention_mask = inputs.attention_mask.to(device)

    # Generate text & word timestamps for THIS SPECIFIC CHUNK ONLY
    with torch.no_grad():
        # FIX 2: Set return_dict_in_generate=True so we can safely unpack outputs
        outputs = model.generate(
            input_features=input_features,
            attention_mask=attention_mask,
            return_timestamps=True, # 'word' for word-level timestamps
            return_dict_in_generate=True, # Wraps outputs in a safe dictionary structure
            temperature=0.0
        )

    # FIX 3: Unpack the generated text token ids cleanly
    predicted_ids = outputs["sequences"]
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    # Append the structured result for this explicit block
    chunked_results.append({
        "chunk_index": len(chunked_results),
        "global_window_seconds": (global_start_time, global_end_time),
        "text": transcription.strip(),
    })

    print(f"Processed Chunk {chunked_results[-1]['chunk_index']}: Window {global_start_time:.1f}s to {global_end_time:.1f}s")

# 6. Inspect your isolated chunk-wise data structure
print("\n--- VIEW OF MANUALLY SEPARATED CHUNKS ---")
import pprint
pprint.pprint(chunked_results)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

...

clean/validation-00000-of-00001-91350812(…):   0%|          | 0.00/1.98M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/1 [00:00<?, ? examples/s]

Total audio length: 62.45 seconds
Processing explicit chunks manually...



[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see re

Processed Chunk 0: Window 0.0s to 30.0s
Processed Chunk 1: Window 25.0s to 55.0s
Processed Chunk 2: Window 50.0s to 62.5s

--- VIEW OF MANUALLY SEPARATED CHUNKS ---
[{'chunk_index': 0,
  'global_window_seconds': (0.0, 30.0),
  'text': 'Mr. Quilter is the apostle of the middle classes, and we are glad '
          "to welcome his gospel. Nor is Mr. Quilter's manner less interesting "
          'than his matter. He tells us that at this festive season of the '
          'year, with Christmas and roast beef looming before us, similes '
          'drawn from eating and its results occur most readily to the mind. '
          "He has grave doubts whether Sir Frederick Layton's work is really "
          'Greek after all, and can discover'},
 {'chunk_index': 1,
  'global_window_seconds': (25.0, 55.0),
  'text': "whether Sir Frederick Layton's work is really Greek after all, and "
          "can discover in it but little of rocky Ithaca. Linnell's pictures "
          "are a sort of Up Guards a